In [6]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260615_154932"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))

snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1781516013114,BTCUSDT,65606.06,65606.07,65606.065,65606.064810,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65609.215,0.000048
1,1781516013214,BTCUSDT,65606.06,65606.07,65606.065,65606.064869,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009
2,1781516013314,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009
3,1781516013414,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65609.355,0.000050,65614.875,0.000134,65606.685,0.000009
4,1781516013514,BTCUSDT,65606.06,65606.07,65606.065,65606.064860,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65614.875,0.000134,65614.875,0.000134,65606.685,0.000009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225582,1781538571914,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225583,1781538572014,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225584,1781538572114,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225585,1781538572214,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
"""

Does ml_delta explain the residual error in my reservation price (mid + struct_delta + micro_signal) | market state?

"""

df = snapshots

horizons = [100, 500, 1000, 5000]

df["reservation_y"] = df["mid"] + df["struct_delta"] + df["micro_signal_delta"] # reconstructed reservation, just to be clear what goes in reservation

for h in horizons:
    df[f"y_{h}ms"] = np.log(df[f"future_mid_{h}ms"] / df["reservation_y"]) # residuals with center = mid + struct_delta + micro_drift

feature_cols = [
    # raw microstructure
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "inventory",
    "volatility",
    "queue_ahead_bid",
    "queue_ahead_ask",
]

df = df.dropna()

split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[feature_cols].to_numpy(dtype=np.float32)
X_test = test[feature_cols].to_numpy(dtype=np.float32)

In [8]:
def ic(pred, y):
    return np.corrcoef(pred, y)[0, 1]

def rank_ic(pred, y):
    return spearmanr(pred, y).statistic

results = {}
models = {}

for h in horizons:

    y_train = train[f"y_{h}ms"]
    y_test = test[f"y_{h}ms"]

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)

    pred = model.predict(X_test)
    true = y_test.values

    # -------------------------
    # signal quality
    # -------------------------
    residual_ic = ic(pred, true)
    residual_rank_ic = rank_ic(pred, true)

    # -------------------------
    # direction accuracy
    # -------------------------
    hit_rate = (np.sign(pred) == np.sign(true)).mean()

    # -------------------------
    # true economic interpretation
    # -------------------------
    pnl_proxy = np.mean(pred * true)
    pnl_std = np.std(pred * true) + 1e-9
    sharpe_proxy = pnl_proxy / pnl_std

    models[h] = {
        "model": model
    }
    results[h] = {
        "Residual_IC": residual_ic,
        "Residual_Rank_IC": residual_rank_ic,
        "HitRate": hit_rate,
        "PnLProxy": pnl_proxy,
        "SharpeProxy": sharpe_proxy,
    }

results_df = pd.DataFrame(results)
results_df

c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\admin\AppData\Local\Temp\ipykernel_312\664318283.py:5: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(pred, y).statistic


,100,500,1000,5000
Residual_IC,NaN,2.720923e-01,2.821073e-01,2.440281e-01
Residual_Rank_IC,NaN,-4.454057e-01,-2.357498e-01,1.865660e-01
HitRate,6.142148e-01,1.777068e-01,2.411102e-01,4.459963e-01
PnLProxy,7.321506e-18,7.902540e-11,2.340853e-10,1.498861e-09
SharpeProxy,7.321335e-09,5.502545e-02,1.075187e-01,1.986935e-01


In [9]:
artifact = {
    "model": models[1000]["model"],
    "feature_cols": feature_cols,
    "target": "log(future_mid/(mid + struct_delta + micro_signal_delta))",
    "horizon_ms": 1000,
}

joblib.dump(artifact, "data/residual_model_3.pkl")

['data/residual_model_3.pkl']

In [10]:
# model = models[1000]["model"]

# model.save_model("data/residual_model_3_xgb.json")

# artifact = {
#     "model": "residual_model_3",
#     "model_file": "residual_model_3_xgb.json",

#     "feature_cols": feature_cols,

#     "target": "log(future_mid/(mid + struct_delta + micro_signal_delta))",

#     "horizon_ms": 1000
# }

# with open("data/residual_model_3.json", "w") as f:
#     json.dump(artifact, f, indent=4)

def export_xgb(model_name, target, horizon_ms):
    model = models[horizon_ms]["model"]
    model.save_model(f"data/{model_name}_xgb.json")

    artifact = {
        "model_name": f"{model_name}",
        "target": target,
        "model_file": f"data/{model_name}_xgb.json",
        "horizon_ms": horizon_ms,
        "feature_cols": feature_cols,
        "feature_dim": len(feature_cols)
    }

    with open(f"data/{model_name}.json", "w") as f:
        json.dump(artifact, f, indent=4)
    
    print(f"['data/{model_name}.json']")
    print(f"['data/{model_name}_xgb.json']")

export_xgb(model_name="residual_model_4", target="log(future_mid/(mid + struct_delta + micro_signal_delta))", horizon_ms=1000)

['data/residual_model_4.json']
['data/residual_model_4_xgb.json']
